In [ ]:
# python  -m matcha.onnx.export /notebooks/bert-vits2/frankenstein-matcha/logs/epoch137.ckpt /tmp/epoch137.onnx \
#     --vocoder-name hifigan_univ_v1 \
#     --vocoder-checkpoint-path /notebooks/bert-vits2/frankenstein-matcha/pretrained_models/g_02500000

# python -m matcha.onnx.infer /tmp/epoch137.onnx \
#     --text "試左，仍然唔識彈出嚟。" \
#     --prompt-audio ref123.wav \
#     --output-dir ./tmp/output

In [ ]:
import onnxruntime as ort
import numpy as np
import torch
import librosa
from matcha.utils.utils import intersperse
from matcha.text import sequence_to_text, text_to_sequence
from matcha.data.text_mel_datamodule import load_spk_embedding, get_spk_embedding
import IPython.display as ipd

# Load ONNX model
model_path = '/tmp/epoch137.onnx'
model = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])

# Load speaker embedding session
speaker_embedding_onnx_session = load_spk_embedding("pretrained_models/campplus.onnx")

def process_text(text: str, jyutping: str, prompt_audio: str, add_blank=True):
    phone_token_ids, tones, word_pos, syllable_pos = text_to_sequence(text, jyutping)
    if add_blank:
        phone_token_ids = intersperse(phone_token_ids, 0)
        tones = intersperse(tones, 0)
        word_pos = intersperse(word_pos, 0)
        syllable_pos = intersperse(syllable_pos, 0)
    x = torch.tensor(phone_token_ids, dtype=torch.long)[None]  # Add batch dim
    tones = torch.tensor(tones, dtype=torch.long)[None]
    word_pos = torch.tensor(word_pos, dtype=torch.long)[None]
    syllable_pos = torch.tensor(syllable_pos, dtype=torch.long)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long)
    prompt_speech = librosa.load(prompt_audio, sr=16000)[0]
    spk_emb = get_spk_embedding(prompt_speech, speaker_embedding_onnx_session)
    spk_emb = torch.tensor(spk_emb, dtype=torch.float)[None]
    return {
        "x": x.numpy(),
        "x_lengths": x_lengths.numpy(),
        "tones": tones.numpy(),
        "word_pos": word_pos.numpy(),
        "syllable_pos": syllable_pos.numpy(),
        "spk_emb": spk_emb.numpy(),
    }

# Example inference
text = "試左，仍然唔識彈出嚟。"
jyutping = "si3 zo2 , jing4 jin4 m4 sik1 daan6 ceot1 lai4 ."  # Assuming no jyutping provided
prompt_audio = "ref123.wav"

processed = process_text(text, jyutping, prompt_audio)

inputs = {
    "x": processed["x"],
    "x_lengths": processed["x_lengths"],
    "scales": np.array([0.667, 1.0], dtype=np.float32),  # temperature, length_scale
    "tone": processed["tones"],
    "word_pos": processed["word_pos"],
    "syllable_pos": processed["syllable_pos"],
    "spk_emb": processed["spk_emb"],
}

# Run inference
outputs = model.run(None, inputs)
wav, wav_lengths = outputs

# Play audio
audio = wav[0][:wav_lengths[0]]
ipd.display(ipd.Audio(audio, rate=22050))